## RAG_Learning

In [1]:
from PyPDF2 import PdfReader


### Read from the pdf and convert to Text

In [15]:
def load_pdf(file):
    reader = PdfReader(file)
    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""
    return text
text = load_pdf(r"C:\Users\mikun\Downloads\5gc2qerb1v_2024_living_planet_report_a_system_in_peril.pdf")

In [30]:
print(text)

WWF LIVING PLANET REPORT 20241
A System in PerilA System in Peril2024LIVINGPLANETREPORT
A System in PerilWWF LIVING PLANET REPORT 20242
 
WWF 
WWF is an independent conservation organisation,  
with more than 38 million followers and a global network  
active through local leadership in over 100 countries.
Our mission is to stop the degradation of the planet’s 
natural environment and to build a future in which people 
live in harmony with nature, by conserving the world’s 
biological diversity, ensuring that the use of renewable 
natural resources is sustainable, and promoting the 
reduction of pollution and wasteful consumption.
ZSL (Zoological Society of London) Institute of Zoology
  
Founded in 1826, ZSL is an international conservation 
charity, driven by science, working to restore wildlife in the 
UK and around the world; by protecting critical species, 
restoring ecosystems, helping people and wildlife live 
together and inspiring support for nature. Through our 
leading conse

### Convert the text to Chunk

In [29]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(text)
print(chunks)

['WWF LIVING PLANET REPORT 20241\nA System in PerilA System in Peril2024LIVINGPLANETREPORT\nA System in PerilWWF LIVING PLANET REPORT 20242\n \nWWF \nWWF is an independent conservation organisation,  \nwith more than 38 million followers and a global network  \nactive through local leadership in over 100 countries.\nOur mission is to stop the degradation of the planet’s \nnatural environment and to build a future in which people \nlive in harmony with nature, by conserving the world’s \nbiological diversity,', 'uture in which people \nlive in harmony with nature, by conserving the world’s \nbiological diversity, ensuring that the use of renewable \nnatural resources is sustainable, and promoting the \nreduction of pollution and wasteful consumption.\nZSL (Zoological Society of London) Institute of Zoology\n  \nFounded in 1826, ZSL is an international conservation \ncharity, driven by science, working to restore wildlife in the \nUK and around the world; by protecting critical species, 

### Create Embeddings

In [20]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_chunks(chunks):
    return embed_model.encode(chunks)
embedding = embed_chunks(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [31]:
print(embedding)

[[ 0.05064786 -0.01150905 -0.01505116 ... -0.08211366 -0.02189469
  -0.0054176 ]
 [-0.01997442  0.10846406 -0.01120047 ... -0.09984788  0.05342874
   0.04044391]
 [ 0.05775991  0.06382176  0.0225799  ... -0.08176004  0.03766115
   0.07679226]
 ...
 [ 0.03300827  0.1032024  -0.03875544 ... -0.12102491 -0.01580573
   0.02370433]
 [ 0.0904241   0.05953547  0.01374167 ... -0.09689687  0.04137389
   0.02001138]
 [ 0.02085897 -0.01779489 -0.00139008 ... -0.08622465  0.04745837
  -0.02957869]]


### Storing in FAISS (Vector DB)

In [21]:
import faiss
import numpy as np

def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings))
    return index
index = create_faiss_index(embedding)

In [32]:
print(index)

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001D2D7506C40> >


In [23]:
def retrieve(query, k, index, chunks):
    query_vec = embed_model.encode([query])
    distances, indices = index.search(query_vec, k)

    results = [chunks[i] for i in indices[0]]
    return results
result = retrieve(query="summarise the pdf", k=5, index=index, chunks=chunks)

In [25]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

llm = genai.GenerativeModel("gemini-flash-latest")

def ask_gemini(context, query):
    prompt = f"""
    Answer ONLY using the context below.
    If the answer is not present, say "Not found in document".

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.generate_content(prompt)
    return response.text
ask_gemini(result, "summarise the pdf")

'The WWF Living Planet Report 2024 defines biodiversity as variability within species and ecosystems, which includes genetic, population, and ecosystem diversity. This biodiversity has direct and indirect effects on quality of life, referred to as “nature’s contributions to people.” The report utilizes the Living Planet Index (LPI) to monitor global animal population trends—which may be increasing, decreasing, or stable—to help understand the health of ecosystems.'

In [27]:
def rag_pipeline(query, index, chunks):
    retrieved_chunks = retrieve(query, 3, index, chunks)
    context = " ".join(retrieved_chunks)

    answer = ask_gemini(context, query)
    return answer
rag_pipeline("summarise the pdf", index, chunks)

'The provided text defines biodiversity as the variability within species and ecosystems, which directly and indirectly affects human quality of life. It details two specific forms of diversity:\n\n*   **Genetic diversity:** The variation of genetic information within populations or species, which is essential for adaptability and long-term persistence.\n*   **Ecosystem diversity:** The variety of terrestrial, marine, and aquatic ecosystems (such as forests and coral reefs) that support a wide range of species and ecological functions.\n\nThe document also outlines reproduction rights, stating that non-commercial use is authorized with written notification to WWF, while commercial use and photo reproduction require prior written permission.'